# KW_MMDS - Colab 2
## Frequent Pattern Mining in Spark

### Setup

Let's setup Spark on your Colab environment.  Run the cell below!

In [ ]:
!pip install pyspark
!pip install -U -q PyDrive
!apt install openjdk-8-jdk-headless -qq
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 38 not upgraded.
Need to get 39.6 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .

Now we authenticate a Google Drive client to download the file we will be processing in our Spark job.

**Make sure to follow the interactive instructions.**

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
id='1dhi1F78ssqR8gE6U-AgB80ZW7V_9snX4'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('products.csv')

id='1KZBNEaIyMTcsRV817us6uLZgm-Mii8oU'
downloaded = drive.CreateFile({'id': id})
downloaded.GetContentFile('order_products__train.csv')

If you executed the cells above, you should be able to see the dataset we will need for this Colab under the "Files" tab on the left panel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import pyspark
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark import SparkContext, SparkConf

Let's initialize the Spark context.

In [ ]:
# create the session
conf = SparkConf().set("spark.ui.port", "4050")

# create the context
sc = pyspark.SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()

### Your task

If you run successfully the setup stage, you are ready to work with the **3 Million Instacart Orders** dataset. In case you want to read more about it, check the [official Instacart blog post](https://tech.instacart.com/3-million-instacart-orders-open-sourced-d40d29ead6f2) about it, a concise [schema description](https://gist.github.com/jeremystan/c3b39d947d9b88b3ccff3147dbcf6c6b) of the dataset, and the [download page](https://www.instacart.com/datasets/grocery-shopping-2017).

In this Colab, we will be working only with a small training dataset (~131K orders) to perform fast Frequent Pattern Mining with the FP-Growth algorithm.

In [ ]:
products = spark.read.csv('products.csv', header=True, inferSchema=True)
orders = spark.read.csv('order_products__train.csv', header=True, inferSchema=True)

In [ ]:
products.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)



In [ ]:
orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



Use the Spark Dataframe API to join 'products' and 'orders', so that you will be able to see the product names in each transaction (and not only their ids).  Then, group by the orders by 'order_id' to obtain one row per basket (i.e., set of products purchased together by one customer).

In this Colab we will explore [MLlib](https://spark.apache.org/mllib/), Apache Spark's scalable machine learning library. Specifically, you can use its implementation of the [FP-Growth](https://spark.apache.org/docs/latest/ml-frequent-pattern-mining.html#fp-growth) algorithm to perform efficiently Frequent Pattern Mining in Spark.
Use the Python example in the documentation, and train a model with

```minSupport=0.01``` and ```minConfidence=0.5```



1. (25점) Compute how many frequent itemsets and association rules were generated by running FP-growth.
- 중요: num_freqItemsets1에는 빈발 항목집합의 개수를 저장하고, num_associationRules1에는 연관 규칙의 개수를 저장합니다.

In [ ]:
# 주석 삭제 금지
# 1번 문제의 코드는 현재 셀에만 작성하세요. (products와 orders의 join도 여기에서 수행)
# 최종 결과는 num_freqItemsets1과 num_associationRules1에 기록합니다.

from pyspark.ml.fpm import FPGrowth

orders_joined = orders.join(products, "product_id")
orders_joined.select("order_id", "product_name")
transactions = orders_joined.groupBy("order_id").agg(collect_list("product_name").alias("items"))

fPGrowth = FPGrowth(itemsCol = "items", minSupport= 0.01, minConfidence = 0.5)

model = fPGrowth.fit(transactions)

freqItemset = model.freqItemsets
associationRules = model.associationRules

num_freqItemsets1 = freqItemset.count()
num_associationRules1 = associationRules.count()
#print(num_freqItemsets1, num_associationRules1)

2. (25점) What is the most frequent item?
- 중요: most_freqItem에는 가장 많이 등장한 product의 product_name을 string으로 기재합니다.

In [ ]:
# 주석 삭제 금지
# 2번 문제의 코드는 현재 셀에만 작성하세요.
# 최종 결과는 most_freqItem에 string으로 저장합니다.

most_frequent_row = freqItemset.filter(size(col("items"))==1).orderBy(desc("freq")).first()


most_freqItem = most_frequent_row["items"][0]




#print(most_freqItem)

3. (25점) Now retrain the FP-growth model changing only
```minsupport=0.001```
and compute how many frequent itemsets and association rules were generated.
- 중요: num_freqItemsets2에는 빈발 항목집합의 개수를 저장하고, num_associationRules2에는 연관 규칙의 개수를 저장합니다.


In [ ]:
# 주석 삭제 금지
# 3번 문제의 코드는 현재 셀에만 작성하세요.
# 최종 결과는 num_freqItemsets2와 num_associationRules2에 기록합니다.

fPGrowth2 = FPGrowth(itemsCol = "items", minSupport= 0.001, minConfidence = 0.5)
model2 = fPGrowth2.fit(transactions)

freqItemsets2 = model2.freqItemsets
associationRules2 = model2.associationRules
num_freqItemsets2 = freqItemsets2.count()
num_associationRules2 =associationRules2.count()

#print(num_freqItemsets2, num_associationRules2)

4. (25점) Collect all the association rules of the above problem.
- 중요: 3번 문제에서 찾은 모든 연관 규칙을 associationRules에 **confidence의 내림차순**으로 저장합니다.
- 중요: type(associationRules)는 pyspark.sql.dataframe.DataFrame 입니다.
- 중요: associationRules.show(truncate=False)를 수행하면 아래와 같이 출력됩니다. (첫번째 행만 표시)

<img src = "https://drive.google.com/uc?id=10-qoZ60-MyX8SzJYcqXb7jAFgrNHPDvW">

In [ ]:
# 주석 삭제 금지
# 4번 문제의 코드는 현재 셀에만 작성하세요.
# 최종 결과는 associationRules에 저장합니다.

associationRules = model2.associationRules.orderBy(desc("confidence"))


#associationRules.show(truncate=False)

5. (0점) 연관 규칙 살펴보기

In [ ]:
# 주석 삭제 금지
# 코드를 작성할 필요는 없습니다.
# 위에서 찾아낸 연관 규칙들을 종합하면 어떤 것을 알 수 있는지 생각해 보세요.


"""
위에서 나타난 결과들을 분석해보면
1. minSupport를 보면 0.1일떄는 num_associationRules1 = 0.001 일 때는 120개, num_associationRules2 = 11때는 444개의
itemSet의 갯수를 보여준다. 이렇게 후보군(freqItemsets)이 크게 늘어났음에도 불구하고,
confidence > 0.5라는 기준을 통과한 규칙은 고작 11개뿐.

2. 이 11개의 confidence기준을 만족하는 Set들도, banana와 연관이 있어서 사는게 아닌 단지 banana를 많이 사니까 연관이 있어보인다고 생각할 수 있음

--> 이것들만 보면 서로 연관이 없는 무작위의 데이터셋이 아닌가 생각할 수 있음

이때 확인해야 하는게 표에 나타나는 Lift값
lift값이 1을 넘으면 보통 상관관계가 높다고 보는데, 4~5에 수렴하는 값들이 많은걸로 보아 실제로는 특정 물품을 사는사람은
아무런 정보 없이 임의의 고객을 선택했을 때보다 유기농 바나나를 살 확률이 높다는 의미

즉, 정리하면 우리가 공부한 confidence값 만으로는 수많은 데이터셋이 존재하는 현실세계에서는 의미없는 관계처럼 비춰질수도 있다.
그래서 lift같은 여러 도구들을 어떻게 적용할지 데이터분석가가 고민해보면서 , 의미없는 관계처럼 보이는 데이터 안에서 의미를 찾는게 데이터분석의 핵심이라 생각
"""


'\n위에서 나타난 결과들을 분석해보면\n1. minSupport를 보면 0.1일떄는 num_associationRules1 = 0.001 일 때는 120개, num_associationRules2 = 11때는 444개의\nitemSet의 갯수를 보여준다. 이렇게 후보군(freqItemsets)이 크게 늘어났음에도 불구하고,\nconfidence > 0.5라는 기준을 통과한 규칙은 고작 11개뿐.\n\n2. 이 11개의 confidence기준을 만족하는 Set들도, banana와 연관이 있어서 사는게 아닌 단지 banana를 많이 사니까 연관이 있어보인다고 생각할 수 있음\n\n--> 이것들만 보면 서로 연관이 없는 무작위의 데이터셋이 아닌가 생각할 수 있음\n\n이때 확인해야 하는게 표에 나타나는 Lift값\nlift값이 1을 넘으면 보통 상관관계가 높다고 보는데, 4~5에 수렴하는 값들이 많은걸로 보아 실제로는 특정 물품을 사는사람은\n아무런 정보 없이 임의의 고객을 선택했을 때보다 유기농 바나나를 살 확률이 높다는 의미\n\n즉, 정리하면 우리가 공부한 confidence값 만으로는 수많은 데이터셋이 존재하는 현실세계에서는 의미없는 관계처럼 비춰질수도 있다.\n그래서 lift같은 여러 도구들을 어떻게 적용할지 데이터분석가가 고민해보면서 , 의미없는 관계처럼 보이는 데이터 안에서 의미를 찾는게 데이터분석의 핵심이라 생각\n'